# Validation Slice Probe

This notebook exercises the first local assertion-validation slice in `onto-canon6` against real donor profiles from `onto-canon5`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from onto_canon6.ontology_runtime import load_profile, validate_assertion_payload  # noqa: E402

In [2]:
def summarize(profile_name: str, version: str, payload: dict[str, object]) -> dict[str, object]:
    profile = load_profile(profile_name, version)
    outcome = validate_assertion_payload(payload, profile=profile)
    return {
        "profile": profile_name,
        "mode": profile.ontology_policy.mode,
        "proposal_policy": profile.ontology_policy.proposal_policy,
        "hard": [finding.code for finding in outcome.hard_errors],
        "soft": [finding.code for finding in outcome.soft_violations],
        "proposals": [proposal.value for proposal in outcome.proposal_requests],
        "type_checks_total": outcome.type_checks_total,
        "type_checks_unknown": outcome.type_checks_unknown,
    }

In [3]:
unknown_predicate_payload = {
    "predicate": "oc:unknown_predicate_demo",
    "roles": {
        "subject": [
            {
                "entity_id": "ent:subject:1",
                "entity_type": "oc:person",
            }
        ]
    },
}

[
    summarize("default", "1.0.0", unknown_predicate_payload),
    summarize("dodaf", "0.1.0", unknown_predicate_payload),
    summarize("psyop_seed", "0.1.0", unknown_predicate_payload),
]

[{'profile': 'default',
  'mode': 'open',
  'proposal_policy': 'reject',
  'hard': [],
  'soft': [],
  'proposals': [],
  'type_checks_total': 0,
  'type_checks_unknown': 0},
 {'profile': 'dodaf',
  'mode': 'closed',
  'proposal_policy': 'reject',
  'hard': ['oc:profile_unknown_predicate'],
  'soft': [],
  'proposals': [],
  'type_checks_total': 0,
  'type_checks_unknown': 0},
 {'profile': 'psyop_seed',
  'mode': 'mixed',
  'proposal_policy': 'allow',
  'hard': [],
  'soft': ['oc:profile_unknown_predicate'],
  'proposals': ['oc:unknown_predicate_demo'],
  'type_checks_total': 0,
  'type_checks_unknown': 0}]

In [4]:
dodaf_valid_payload = {
    "predicate": "dodaf:service_supports_activity",
    "roles": {
        "service": [
            {
                "entity_id": "ent:service:1",
                "entity_type": "dm2:CoalitionService",
            }
        ],
        "activity": [
            {
                "entity_id": "ent:activity:1",
                "entity_type": "dm2:OperationalActivity",
            }
        ],
    },
}

summarize("dodaf", "0.1.0", dodaf_valid_payload)

{'profile': 'dodaf',
 'mode': 'closed',
 'proposal_policy': 'reject',
 'hard': [],
 'soft': [],
 'proposals': [],
 'type_checks_total': 2,
 'type_checks_unknown': 0}

In [5]:
psyop_value_kind_error_payload = {
    "predicate": "oc:hold_command_role",
    "roles": {
        "commander": [
            {
                "entity_id": "ent:person:olson",
                "entity_type": "oc:person",
            }
        ],
        "organization": [
            {
                "entity_id": "ent:org:ussocom",
                "entity_type": "oc:military_organization",
            }
        ],
        "role_title": [
            {
                "kind": "value",
                "value_kind": "time",
            }
        ],
    },
}

summarize("psyop_seed", "0.1.0", psyop_value_kind_error_payload)

{'profile': 'psyop_seed',
 'mode': 'mixed',
 'proposal_policy': 'allow',
 'hard': ['oc:profile_role_value_kind_violation'],
 'soft': [],
 'proposals': [],
 'type_checks_total': 2,
 'type_checks_unknown': 0}